# 📓 Semana 14 · Dia 5 — Avaliação de agentes e testes de regressão

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Suite de testes do agente |

---


## 📖 Teoria — Avaliar agentes ≠ avaliar RAG

Além das métricas de RAG, agentes têm: **tool correctness** (a tool certa foi chamada?), **SQL correctness** (o SQL estava certo?) e **recall de tool**.

O `mlflow.evaluate` com `model_type='databricks-agent'` cobre isso com LLM-as-judge + tracing automático.


### 💻 Na prática — Golden set do agente

Crie perguntas com a tool/SQL esperado.


In [ ]:
# Golden set do agente
import pandas as pd
agente_golden = pd.DataFrame({
    "question": ["Qual a receita do UK?", "Top 3 produtos?"],
    "expected_tool": ["receita_por_pais", "top_produtos"],
    "expected_response": ["um número", "lista de produtos"]
})
print(agente_golden)

### 💻 Na prática — Rodando a avaliação

Execute o agente no golden set e avalie.


In [ ]:
# Executar + avaliar
import mlflow
respostas = []
for q in agente_golden["question"]:
    respostas.append(executor.invoke({"input": q})["output"])
agente_golden["response"] = respostas
with mlflow.start_run(run_name="avaliacao_agente_v1"):
    mlflow.evaluate(
        data=agente_golden[["question", "response"]],
        targets=agente_golden["expected_response"],
        model_type="databricks-agent",
        extra_metrics=[mlflow.metrics.genai.faithfulness(),
                       mlflow.metrics.genai.answer_relevance()])
print("Avaliação do agente registrada (com traces).")

### 💻 Na prática — Testes de regressão

Rode o golden set a cada mudança (prompt/tool) — como CI de agente.


In [ ]:
# Mini suíte de regressão
def testar_regressao():
    falhas = []
    for q, tool_esperada in zip(agente_golden["question"], agente_golden["expected_tool"]):
        r = executor.invoke({"input": q})
        if tool_esperada not in str(r.get("intermediate_steps", [])):
            falhas.append(q)
    return falhas
print("Falhas de regressão:", testar_regressao() or "nenhuma")

> 🎯 **Dica de prova**: Agentes: avalie tool correctness + resposta. Pergunta: 'o que muda na avaliação de um agente vs RAG?' → verificar se a tool certa foi chamada (traces).


## 🎯 Exercícios de fixação

**1.** Adicione 5 perguntas novas ao golden set do agente.

**2.** Por que rodar regressão a cada mudança?

**3.** O que um trace de agente mostra que o de RAG não?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Golden set

Cubra: receita por país, top produtos, comparação de meses, país inexistente (erro).

**2.** Regressão

Mudou o prompt/tool → comportamento pode mudar; a suíte pega a regressão antes da produção.

**3.** Trace de agente

As chamadas de tool (nome, args, resultado) e a decisão do LLM — o 'caminho' do raciocínio.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*